In [1]:
"""
S01 MPO training — integrate notebooks 01–07 mission context.

Scenario:
  - Same mission profile as notebook 07 baseline overflight (50-target meridian grid,
    seeded clouds over the corridor, dual camera, attitude safety on, image quality)
  - Controller features selected via ControllerFeatureConfig (see cell below)
  - 10× baseline overflight warmup (notebook 07 policy, 2-D buffer) → train → eval
  - Preflight: inline feature checks + full ML training pytest suite

Verification: s01_utils/training_workflow.py
Artifacts: autonomous_control/models/nb-s01-08-<timestamp>/
Export: eval_best.mp4 in run directory
"""

'\nS01 MPO training — integrate notebooks 01–07 mission context.\n\nScenario:\n  - Same mission profile as notebook 07 baseline overflight (50-target meridian grid,\n    seeded clouds over the corridor, dual camera, attitude safety on, image quality)\n  - Controller features selected via ControllerFeatureConfig (see cell below)\n  - 10× baseline overflight warmup (notebook 07 policy, 2-D buffer) → train → eval\n  - Preflight: inline feature checks + full ML training pytest suite\n\nVerification: s01_utils/training_workflow.py\nArtifacts: autonomous_control/models/nb-s01-08-<timestamp>/\nExport: eval_best.mp4 in run directory\n'

In [2]:
def setup_notebook_paths():
    """
    Configure Python paths and working directory for running the S01 training notebook.

    - Walks up directories from the current working directory until it finds the 'simulation' folder,
      which marks the backend root.
    - Changes the working directory to the backend root to ensure relative paths are correct.
    - Adds both the backend root and the S01 notebook utilities directory to sys.path for imports.

    This setup is required for importing backend modules and utility code in other cells.
    """
    import os
    import sys
    from pathlib import Path

    notebook_dir = Path.cwd()
    backend_root = notebook_dir
    for _ in range(6):
        if (backend_root / "simulation").is_dir():
            break
        backend_root = backend_root.parent
    os.chdir(backend_root)
    sys.path.insert(0, str(backend_root))
    _s01_dir = backend_root / "notebooks" / "s01"
    sys.path.insert(0, str(_s01_dir))
    print(f"backend_root={backend_root}")

setup_notebook_paths()

backend_root=/home/cedric/code/auto-sat-control/backend


In [3]:
import importlib

import s01_utils.training_workflow as tw

importlib.reload(tw)

# Fast gate: inline checks + unit pytest (~3s). Re-runs are skipped via session/disk cache.
# For full serial episode-runner tests (~80s): tw.run_s01_training_preflight_gate(integration_pytest=True, force=True)
tw.run_s01_training_preflight_gate()

/home/cedric/miniforge3/envs/auto-sat/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Using device: cuda
Using device: cuda


True

In [ ]:
from dataclasses import replace

# Single source of truth: s01_utils/training_workflow.py (timestep keys + mission scalars).
# Customize with replace(), e.g. replace(tw.S01_TRAINING_FEATURE_CONFIG, include_capture_budget=False)
FEATURE_CONFIG = tw.S01_TRAINING_FEATURE_CONFIG

WORKFLOW_CONFIG = tw.TrainingWorkflowConfig(
    seed=7,
    train_episodes=20,
    feature_config=FEATURE_CONFIG,
)

#TODO
#try the following:
# give the image quality / latent reward to the controller
# this should help it distinguish between good and bad images, and learn to avoid bad images (for shutter)

# Mission profile for layout tables (same as training).
_mission_resolved = tw.build_s01_training_mission_setup(seed=WORKFLOW_CONFIG.seed).resolve(
    require_camera=True
)
_secondary_bins = int(_mission_resolved.secondary_camera_observation_line_n_bins)
_n_targets = len(_mission_resolved.target_areas or ())
tw.display_feature_tables(
    FEATURE_CONFIG,
    secondary_camera_bins=_secondary_bins,
    n_mission_targets=_n_targets,
)

### Controller feature selection (`ControllerFeatureConfig`)

Edit `S01_TRAINING_FEATURE_CONFIG` in `s01_utils/training_workflow.py` (timestep key tuples **and** `include_capture_budget` / `include_target_bearing_errors`). Notebook 08 should assign `FEATURE_CONFIG = tw.S01_TRAINING_FEATURE_CONFIG` rather than duplicating keys.

,group,timestep_key,state_dims,unit,encoder_path,source_module
0,attitude,body_z_angle_rad,1,rad,scalar → MLP branch,autonomous_control/feature_selection.py
1,attitude,omega_sat_rad_s,1,rad/s,scalar → MLP branch,autonomous_control/feature_selection.py
2,orbit,theta_orbit_rad,1,rad,scalar → MLP branch,autonomous_control/feature_selection.py
3,vision,camera_observation_line_codes,100,obs code / bin,int8 codes → embed → 1D-CNN,autonomous_control/feature_selection.py
4,vision,secondary_camera_observation_line_codes,200,obs code / bin,int8 codes → embed → 1D-CNN,autonomous_control/feature_selection.py
5,mission,capture_budget_remaining,1,count,scalar → MLP branch,autonomous_control/controller_observation.py
6,mission,target_bearing_error_rad_0,1,rad,scalar → MLP branch,autonomous_control/controller_observation.py
7,mission,target_bearing_error_rad_1,1,rad,scalar → MLP branch,autonomous_control/controller_observation.py
8,mission,target_bearing_error_rad_2,1,rad,scalar → MLP branch,autonomous_control/controller_observation.py
9,mission,target_bearing_error_rad_3,1,rad,scalar → MLP branch,autonomous_control/controller_observation.py


### Controller encoder routing (scalar **54**, vision streams **2**)

,stage,inputs,input_shape,module
0,scalar branch,"body_z_angle_rad, omega_sat_rad_s, theta_orbit...","(54,) float32",MLP → 90-D
1,vision branch (primary (nadir)),camera_observation_line_codes,"(100,) int8",ObservationLineCNNEncoder → 32-D
2,vision branch (secondary (forward)),secondary_camera_observation_line_codes,"(200,) int8",ObservationLineCNNEncoder → 32-D
3,vision fusion,concat CNN embeddings,"(64,)",MLP → 90-D
4,policy / Q trunk,"concat(scalar, vision)","(180,)",Actor head / Critic head


### Observation code legend (vision line bins)

,code,label
0,-99,not_computed
1,0,space
2,1,earth
3,2,cloud
4,3,target


In [ ]:
setup = tw.build_training_workflow_setup(WORKFLOW_CONFIG)
tw.print_training_setup_summary(setup)
tw.display_feature_snapshot_tables(setup)

#TODO reduce the number of prints or write to log or run in terminal (is unreadable in the python notebook just too many prints)

Using device: cuda
S01 MPO training setup (notebook 07 baseline overflight profile)
  run_dir:           /home/cedric/code/auto-sat-control/backend/autonomous_control/models/nb-s01-08-2026-06-26_11-18-45
  seed:              7
  altitude:          528.8 km
  targets:           50
  clouds:            25 (seeded over target corridor)
  orbit window:      -32.7° .. 37.1°
  episode steps:     2904
  attitude safety:   on (training_episode_simulation_config)
  scalar dim:        54
  vision streams:    [('camera_observation_line_codes', 100), ('secondary_camera_observation_line_codes', 200)]
  encoder trunk:     180-D
  secondary bins:    200
  feature keys:      ['body_z_angle_rad', 'omega_sat_rad_s', 'theta_orbit_rad', 'camera_observation_line_codes', 'secondary_camera_observation_line_codes']
  warmup episodes:   10
  train episodes:    20
  eval episodes:     2
  MPO batch_size:    256
  MPO gamma:         0.99
  MPO LRs q/pi/eta:  0.00045/0.00015/0.001
  target_kl mu/sigma: 0.1/0.0001

/home/cedric/code/auto-sat-control/backend/simulation/stepper.py:171: UserWarning: controller_update_interval (1 s) is not an integer multiple of simulation_timestep (0.4 s); using nearest multiple: 0.8 s (2 sim steps).
  self._controller_interval_steps, self._effective_controller_interval_s = resolve_controller_interval_steps(


### Scalar features at episode start (MLP branch)

,scalar_index,group,timestep_key,value,unit
0,0,attitude,body_z_angle_rad,4.111343,rad
1,1,attitude,omega_sat_rad_s,0.000000,rad/s
2,2,orbit,theta_orbit_rad,0.969750,rad
3,3,mission,capture_budget_remaining,10.000000,count
4,4,mission,target_bearing_error_rad_0,-1.175578,rad
5,5,mission,target_bearing_error_rad_1,-1.174900,rad
6,6,mission,target_bearing_error_rad_2,-1.174086,rad
7,7,mission,target_bearing_error_rad_3,-1.173142,rad
8,8,mission,target_bearing_error_rad_4,-1.172076,rad
9,9,mission,target_bearing_error_rad_5,-1.170894,rad


### Vision line features at episode start (CNN branches)

,camera,timestep_key,n_bins,preview,dominant_code,target_bins
0,primary (nadir),camera_observation_line_codes,100,"[1, 1, 1, 1, 1, 1, 1, 1, …]",1 (earth),0
1,secondary (forward),secondary_camera_observation_line_codes,200,"[1, 1, 1, 1, 1, 1, 1, 1, …]",1 (earth),0


Structured `ControllerObservation`: scalars **54**, vision **camera_observation_line_codes 100 bins, secondary_camera_observation_line_codes 200 bins** → encoder trunk **180**-D.

In [ ]:
result = tw.run_training_workflow(setup, show_progress=True)
#TODO start a video export sometimes (when reward improved, this code now ran 54 minutes but did not produce one evidence video this is a bit sad)

Warmup:   0%|          | 0/10 [00:00<?, ?ep/s]

╭───────────────────────────────────────────── Simulation info ──────────────────────────────────────────────╮
│  Parameter                                     Value                                                       │
│  episode duration                              1160.93 s                                                   │
│  simulation timestep                           0.399906 s                                                  │
│  integration steps                             2903 (+1 state samples)                                     │
│  orbit altitude                                528.76 km                                                   │
│  orbit period                                  5703.8 s                                                    │
│  theta center offset                           90.00 deg                                                   │
│  episode theta start (rel. center)             -32.693 deg                                                 │
│  episode theta end (rel. center)               37.091 deg                                                  │
│  sat motion span scale                         1.050                                                       │
│  sat z offset                                  0.00 deg                                                    │
│  target areas                                  50                                                          │
│  target phi stripe                             79.93 deg .. 104.46 deg                                     │
│  cloud patches                                 25                                                          │
│  render mode                                   headless                                                    │
│  torque command source                         external                                                    │
│  torque policy                                 sequential_target_baseline                                  │
│  attitude controller                           enabled                                                     │
│  control stack (display)                       sequential_target_baseline · attitude_controller            │
│  controller seed                               -                                                           │
│  controller update (configured)                1 s                                                         │
│  controller update (effective)                 0.8 s (2 steps)                                             │
│  reaction-wheel torque max                     0.1000 N*m                                                  │
│  attitude safety cutoff                        |omega_sat| > 3.00 deg/s -> block opposing torque           │
│  camera kernel backend                         accelerated                                                 │
│  reward shaping                                outer_gate                                                  │
│  episode runner mode                           warmup                                                      │
│  agent                                         MPOAgent                                                    │
│  Camera 1 (primary) - role                     shapes reward (observation line + strip)                    │
│  Camera 1 (primary) - tilt off nadir           0.00 deg                                                    │
│  Camera 1 (primary) - FOV (cross x along)      1.61 deg x 1.20 deg                                         │
│  Camera 1 (primary) - resolution               9344 x 7000 px                                              │
│  Camera 1 (primary) - pixel pitch              3.20 um                                                     │
│  Camera 1 (primary) - focal length             1067.0 mm                                                   │
│  Camera 1 (primary) - GSD @ 528.8 km           1.586 m                                                     │
│  C

╭──────────────────────────────────────────────── Live stats ────────────────────────────────────────────────╮
│  Parameter       Value                                                                                     │
│  episode         warmup ep 1 / 10                                                                          │
│  step            400 / 2903 (13.8%)                                                                        │
│  sim time        160.0 s                                                                                   │
│  step reward     0.0000                                                                                    │
│  episode return  0.00                                                                                      │
│  body z angle    -1.9956 rad                                                                               │
│  omega sat       0.0011 rad/s                                                                              │
│  image smear     0.442 px                                                                                  │
│  image quality   0.2998                                                                                    │
│  buffer          399                                                                                       │
│  agent steps     399                                                                                       │
╰────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Live stats ────────────────────────────────────────────────╮
│  Parameter       Value                                                                                     │
│  episode         warmup ep 1 / 10                                                                          │
│  step            800 / 2903 (27.6%)                                                                        │
│  sim time        319.9 s                                                                                   │
│  step reward     0.0000                                                                                    │
│  episode return  0.00                                                                                      │
│  body z angle    -2.0275 rad                                                                               │
│  omega sat       -0.0348 rad/s                                                                             │
│  image smear     1.694 px                                                                                  │
│  image quality   0.0985                                                                                    │
│  buffer          799                                                                                       │
│  agent steps     799                                                                                       │
╰────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Live stats ────────────────────────────────────────────────╮
│  Parameter       Value                                                                                     │
│  episode         warmup ep 1 / 10                                                                          │
│  step            1200 / 2903 (41.3%)                                                                       │
│  sim time        479.9 s                                                                                   │
│  step reward     0.0000                                                                                    │
│  episode return  140.65                                                                                    │
│  body z angle    -1.5615 rad                                                                               │
│  omega sat       0.0014 rad/s                                                                              │
│  image smear     0.431 px                                                                                  │
│  image quality   0.3043                                                                                    │
│  buffer          1199                                                                                      │
│  agent steps     1199                                                                                      │
╰────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Live stats ────────────────────────────────────────────────╮
│  Parameter          Value                                                                                  │
│  episode            train ep 9 / 20                                                                        │
│  step               1200 / 2903 (41.3%)                                                                    │
│  sim time           479.9 s                                                                                │
│  step reward        0.0000                                                                                 │
│  episode return     0.00                                                                                   │
│  body z angle       -1.6432 rad                                                                            │
│  omega sat          0.0011 rad/s                                                                           │
│  image smear        0.442 px                                                                               │
│  image quality      0.2999                                                                                 │
│  buffer             50000                                                                                  │
│  agent steps        53453                                                                                  │
│  exploration        active (policy sample)                                                                 │
│  kl (ep mean)       1955500.748853                                                                         │
│  q loss (ep mean)   0.373867                                                                               │
│  pi loss (ep mean)  -0.869828                                                                              │
│  eta (ep mean)      28551728332800.000000                                                                  │
╰────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Live stats ────────────────────────────────────────────────╮
│  Parameter          Value                                                                                  │
│  episode            train ep 9 / 20                                                                        │
│  step               1300 / 2903 (44.8%)                                                                    │
│  sim time           519.9 s                                                                                │
│  step reward        0.0000                                                                                 │
│  episode return     0.00                                                                                   │
│  body z angle       -1.2762 rad                                                                            │
│  omega sat          0.0460 rad/s                                                                           │
│  image smear        1.225 px                                                                               │
│  image quality      0.1277                                                                                 │
│  buffer             50000                                                                                  │
│  agent steps        53553                                                                                  │
│  exploration        active (policy sample)                                                                 │
│  kl (ep mean)       2078431.686008                                                                         │
│  q loss (ep mean)   0.365291                                                                               │
│  pi loss (ep mean)  -0.869827                                                                              │
│  eta (ep mean)      28551728332800.000000                                                                  │
╰────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

train ep 9:  46%|████▌     | 1321/2903 [02:32<03:02,  8.69step/s, reward=0.000, total=0.0]

Train:  40%|████      | 8/20 [43:45<1:05:38, 328.25s/ep, kl=1465893.6147, reward=0.0]

KeyboardInterrupt: 

In [ ]:
tw.print_training_kpis(result)

In [ ]:
from utils.notebook.video import init_video_cell, play_saved_video

init_video_cell()

from utils.notebook.video import init_video_cell, play_saved_video

init_video_cell()
tw.display_training_artifacts(result)
video_path = result.artifact_paths["eval_best_video"]
if video_path.exists():
    play_saved_video(video_path)